# Логистическая регрессия и показатели ошибок классификации
![LogRegr.png](Assets/LogRegr.png)


# Цели обучения

- Описать логистическую регрессию и её отличия от линейной регрессии
- Определить метрики ошибок классификации и сценарии, в которых их можно использовать
- Применять расширение Intel® для Scikit-learn*, чтобы использовать вычислительные возможности аппаратного обеспечения



## Введение

Мы будем использовать базу данных [«Распознавание человеческой активности с помощью смартфонов»](https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones), которая была сформирована на основе записей участников исследования, выполнявших повседневные действия (ADL) с смартфоном, оснащенным встроенными инерционными датчиками. Цель состоит в классификации действий по одному из шести видов (ходьба, подъем по лестнице, спуск по лестнице, сидение, стояние и лежание).

Для каждой записи в наборе данных предоставляются следующие данные:

- Трехосное ускорение, полученное с акселерометра (общее ускорение), и расчетное ускорение тела.
- Трехосевая угловая скорость, полученная с гироскопа.
- Вектор из 561 признака с переменными во временной и частотной областях.
- Метка вида деятельности.

Более подробная информация о признаках доступна на указанном выше веб-сайте.

In [ ]:
from __future__ import print_function
import os
data_path = [ 'data']

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from sklearnex import patch_sklearn
patch_sklearn()

## Вопрос 1

Импортируйте данные и выполните следующие действия:

* Проанализируйте типы данных — столбцов много, поэтому целесообразно использовать подсчёт значений
* Определите, требуется ли масштабирование значений с плавающей запятой
* Определите структуру каждого вида деятельности
* Кодируйте название вида деятельности в виде целого числа

In [ ]:
import pandas as pd
import numpy as np

filepath = os.sep.join(data_path + ['Human_Activity_Recognition_Using_Smartphones_Data.csv'])
data = pd.read_csv(filepath, sep=',')

Все столбцы данных содержат числа типа float, за исключением метки активности.

In [ ]:
data.dtypes.value_counts()

In [ ]:
data.dtypes.tail()

Все данные приведены к шкале от -1 (минимум) до 1,0 (максимум).

In [ ]:
data.iloc[:, :-1].min().value_counts()

In [ ]:
data.iloc[:, :-1].max().value_counts()

Проанализируйте структуру мероприятий — они относительно сбалансированы.

In [ ]:
data.Activity.value_counts()

Классификаторы Scikit-learn не принимают разреженные матрицы в качестве столбца для прогнозирования. Поэтому необходимо либо использовать `LabelEncoder` для преобразования меток активности в целые числа, либо, если используется `DictVectorizer`, преобразовать результирующую матрицу в неразреженный массив.
Используйте `LabelEncoder` для преобразования столбца «Activity» с помощью `fit_transform` и просмотрите 5 случайных значений.

In [ ]:
le = LabelEncoder()
data['Activity'] = le.fit_transform(data.Activity)
data['Activity'].sample(5)

## Вопрос 2

* Рассчитайте коэффициенты корреляции между зависимыми переменными.
* Постройте гистограмму значений коэффициентов корреляции.
* Определите, какие из них наиболее сильно коррелируют (положительно или отрицательно).

In [ ]:
# Calculate the correlation values
feature_cols = data.columns[:-1]
corr_values = data[feature_cols].corr()

# Simplify by emptying all the data below the diagonal
tril_index = np.tril_indices_from(corr_values)

# Make the unused values NaNs
for coord in zip(*tril_index):
    corr_values.iloc[coord[0], coord[1]] = np.NaN

# Stack the data and convert to a data frame
corr_values = (corr_values
               .stack()
               .to_frame()
               .reset_index()
               .rename(columns={'level_0':'feature1',
                                'level_1':'feature2',
                                0:'correlation'}))

# Get the absolute values for sorting
corr_values['abs_correlation'] = corr_values.correlation.abs()

Гистограмма корреляций по абсолютным значениям.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
sns.set_context('talk')
sns.set_style('white')
sns.set_palette('dark')

ax = corr_values.abs_correlation.hist(bins=50)

ax.set(xlabel='Absolute Correlation', ylabel='Frequency');

In [ ]:
# The most highly correlated values
corr_values.sort_values('correlation', ascending=False).query('abs_correlation>0.8')

## Вопрос 3

* Разделите данные на обучающий и тестовый наборы. Это можно сделать любым способом, но рекомендуется использовать метод `StratifiedShuffleSplit` из библиотеки Scikit-learn, чтобы сохранить одинаковое соотношение классов предикторов.
* Независимо от метода, использованного для разделения данных, сравните соотношение классов в обучающем и тестовом наборах.


In [ ]:
# Get the split indexes
strat_shuf_split = StratifiedShuffleSplit(n_splits=1,
                                          test_size=0.3,
                                          random_state=42)

train_idx, test_idx = next(strat_shuf_split.split(data[feature_cols], data.Activity))

# Create the dataframes
X_train = data.loc[train_idx, feature_cols]
y_train = data.loc[train_idx, 'Activity']

X_test  = data.loc[test_idx, feature_cols]
y_test  = data.loc[test_idx, 'Activity']

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
y_test.value_counts(normalize=True)

## Вопрос 4

* Постройте модель логистической регрессии без какой-либо регуляризации с использованием всех признаков. Обязательно ознакомьтесь с документацией по построению многоклассовых моделей, чтобы понять, как интерпретировать выведенные коэффициенты. Сохраните модель.
* Используя перекрестную валидацию для определения гиперпараметров, постройте модели с L1- и L2-регуляризацией. Сохраните также каждую из этих моделей. Обратите внимание на ограничения, связанные с многоклассовыми моделями, решателями и видами регуляризации. Построение регуляризованных моделей, в частности модели с L1-регуляризацией, вероятно, займет некоторое время.

In [ ]:
# Standard logistic regression
lr = LogisticRegression(C=.001, max_iter=295).fit(X_train, y_train)

In [ ]:
# L1 regularized logistic regression
lr_l1 = LogisticRegressionCV(Cs=10, cv=4, penalty='l1', solver='liblinear').fit(X_train, y_train)

In [ ]:
# L2 regularized logistic regression
lr_l2 = LogisticRegressionCV(Cs=1, cv=4, penalty='l2').fit(X_train, y_train)

## Вопрос 5

Сравните величины коэффициентов для каждой из моделей. Если использовалась подстановка «один против остальных», каждый набор коэффициентов можно отобразить на графике отдельно.

In [ ]:
# Combine all the coefficients into a dataframe
coefficients = list()

coeff_labels = ['lr', 'l1', 'l2']
coeff_models = [lr, lr_l1, lr_l2]

for lab,mod in zip(coeff_labels, coeff_models):
    coeffs = mod.coef_
    coeff_label = pd.MultiIndex(levels=[[lab], [0,1,2,3,4,5]],
                                codes=[[0,0,0,0,0,0], [0,1,2,3,4,5]])
    coefficients.append(pd.DataFrame(coeffs.T, columns=coeff_label))

coefficients = pd.concat(coefficients, axis=1)

coefficients.sample(10)

Подготовьте шесть отдельных графиков для каждого из многоклассовых коэффициентов.

In [ ]:
fig, axList = plt.subplots(nrows=3, ncols=2)
axList = axList.flatten()
fig.set_size_inches(10,10)

for ax in enumerate(axList):
    loc = ax[0]
    ax = ax[1]

    data = coefficients.xs(loc, level=1, axis=1)
    data.plot(marker='o', ls='', ms=2.0, ax=ax, legend=False)

    if ax is axList[0]:
        ax.legend(loc=4)

    ax.set(title='Coefficient Set '+str(loc))

plt.tight_layout()

## Вопрос 6

* Определите и сохраните класс для каждой модели.
* Также сохраните вероятность для предсказанного класса для каждой модели.

In [ ]:
# Predict the class and the probability for each
y_pred = list()
y_prob = list()

coeff_labels = ['lr', 'l1', 'l2']
coeff_models = [lr, lr_l1, lr_l2]

for lab,mod in zip(coeff_labels, coeff_models):
    y_pred.append(pd.Series(mod.predict(X_test), name=lab))
    y_prob.append(pd.Series(mod.predict_proba(X_test).max(axis=1), name=lab))

y_pred = pd.concat(y_pred, axis=1)
y_prob = pd.concat(y_prob, axis=1)

y_pred.head()

In [ ]:
y_prob.head()

## Вопрос 7

Для каждой модели рассчитайте следующие показатели ошибок:

* точность
* прецизионность
* полнота
* F-показатель
* матрица путаницы

Определите, как объединить показатели для нескольких классов в одно значение для каждой модели.

In [ ]:
metrics = list()
cm = dict()

for lab in coeff_labels:

    # Preciision, recall, f-score from the multi-class support function
    precision, recall, fscore, _ = score(y_test, y_pred[lab], average='weighted')

    # The usual way to calculate accuracy
    accuracy = accuracy_score(y_test, y_pred[lab])

    # ROC-AUC scores can be calculated by binarizing the data
    auc = roc_auc_score(label_binarize(y_test, classes=[0,1,2,3,4,5]),
              label_binarize(y_pred[lab], classes=[0,1,2,3,4,5]),
              average='weighted')

    # Last, the confusion matrix
    cm[lab] = confusion_matrix(y_test, y_pred[lab])

    metrics.append(pd.Series({'precision':precision, 'recall':recall,
                              'fscore':fscore, 'accuracy':accuracy,
                              'auc':auc},
                             name=lab))

metrics = pd.concat(metrics, axis=1)

In [ ]:
metrics

## Вопрос 8

Отобразите или постройте матрицу путаницы для каждой модели.

In [ ]:
fig, axList = plt.subplots(nrows=2, ncols=2)
axList = axList.flatten()
fig.set_size_inches(12, 10)

axList[-1].axis('off')

for ax,lab in zip(axList[:-1], coeff_labels):
    sns.heatmap(cm[lab], ax=ax, annot=True, fmt='d');
    ax.set(title=lab);

plt.tight_layout()

In [ ]:
#493.98 sec